In [0]:
import re
from pyspark.sql import functions as F

## Raw file ingestion

In [0]:
raw_data_dir = "/Volumes/asb/olist_bronze/data_raw"
files = dbutils.fs.ls(raw_data_dir)
files

In [0]:
def clean_table_name(file_name: str) -> str:
    cleaned = file_name.replace(".csv", "").lower()
    cleaned = re.sub(r"[^\w]+", "_", cleaned)
    return re.sub(r"_+", "_", cleaned).strip("_")

for file_info in files:
    if file_info.name.endswith(".csv"):
        raw_table_base = clean_table_name(file_info.name)
        bronze_table = f"asb.olist_bronze.raw_{raw_table_base}"
        
        df_raw = (
            spark.read.format("csv")
            .option("header", "true")
            .option("inferSchema", "false")
            .load(file_info.path)
        )
            
        df_bronze = (
            df_raw
            .withColumns({
                "_input_file_name": F.col("_metadata.file_path"),
                "_ingested_at": F.current_timestamp(),
            })
        )
            
        (df_bronze.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(bronze_table)
        )

## Data profiling

Before establishing rules, profile each entity to discover edge cases, null counts, duplicate keys, and range bounds.

In [0]:
%sql
-- Profile Orders: Check for duplicates, unpopulated status, and timestamp reversals
SELECT 
    COUNT(*) AS total_records,
    COUNT(DISTINCT order_id) AS unique_orders,
    COUNT(CASE WHEN customer_id IS NULL THEN 1 END) AS null_customers,
    COUNT(CASE WHEN order_status IS NULL THEN 1 END) AS null_statuses,
    COUNT(CASE WHEN CAST(order_delivered_customer_date AS TIMESTAMP) < CAST(order_purchase_timestamp AS TIMESTAMP) THEN 1 END) AS delivered_before_purchase
FROM asb.olist_bronze.raw_olist_orders_dataset
;

In [0]:
%sql
-- Profile Order Items: Check for zero or negative values and orphaned records
SELECT 
    COUNT(*) AS total_items,
    MIN(CAST(price AS DOUBLE)) AS min_price,
    MAX(CAST(price AS DOUBLE)) AS max_price,
    COUNT(CASE WHEN CAST(price AS DOUBLE) <= 0 THEN 1 END) AS non_positive_prices,
    COUNT(CASE WHEN CAST(freight_value AS DOUBLE) < 0 THEN 1 END) AS negative_freight
FROM asb.olist_bronze.raw_olist_order_items_dataset
;

In [0]:
%sql
-- Profile Payments: Check installment counts and total payment values
SELECT 
    payment_type,
    MIN(CAST(payment_installments AS INT)) AS min_installments,
    MAX(CAST(payment_installments AS INT)) AS max_installments,
    MIN(CAST(payment_value AS DOUBLE)) AS min_payment_value,
    COUNT(CASE WHEN CAST(payment_value AS DOUBLE) <= 0 THEN 1 END) AS zero_or_neg_payments
FROM asb.olist_bronze.raw_olist_order_payments_dataset
GROUP BY payment_type
;

In [0]:
%sql
-- Profile Reviews: Check survey response ranges
SELECT 
    MIN(TRY_CAST(review_score AS INT)) AS min_score,
    MAX(TRY_CAST(review_score AS INT)) AS max_score,
    COUNT(CASE WHEN TRY_CAST(review_score AS INT) IS NULL OR TRY_CAST(review_score AS INT) NOT BETWEEN 1 AND 5 THEN 1 END) AS invalid_scores
FROM asb.olist_bronze.raw_olist_order_reviews_dataset
;

In [0]:
%sql
-- Profile Products: Identify missing dimension attributes
SELECT 
    COUNT(*) AS total_products,
    COUNT(CASE WHEN product_category_name IS NULL THEN 1 END) AS null_categories,
    COUNT(CASE WHEN CAST(product_weight_g AS DOUBLE) <= 0 THEN 1 END) AS invalid_weights
FROM asb.olist_bronze.raw_olist_products_dataset
;

In [0]:
%sql
-- Profile Geolocation: Identify latitude and longitude boundaries in Brazil
-- Brazil geographic bounds: Lat [-34.0, 5.5], Lng [-74.0, -34.0]
SELECT 
    COUNT(DISTINCT geolocation_zip_code_prefix) AS unique_zip_prefixes,
    MIN(CAST(geolocation_lat AS DOUBLE)) AS min_lat,
    MAX(CAST(geolocation_lat AS DOUBLE)) AS max_lat,
    COUNT(CASE WHEN CAST(geolocation_lat AS DOUBLE) > 5.5 OR CAST(geolocation_lat AS DOUBLE) < -34.0 THEN 1 END) AS out_of_bounds_lat
FROM asb.olist_bronze.raw_olist_geolocation_dataset
;